# ESS 118: Midterm Examination — Coding Portion

### Prof. Leonard Ohenhen, UC Irvine, Earth System Science (ESS)

**Instructions**

1. Save this notebook using the format **ESS118_Midterm_LastName_FirstName.ipynb**.
2. Keep all code, figures, and written answers in this notebook.
3. Run all cells in order before submitting.
4. This is the coding portion of the midterm (60 points).

## Overview

This midterm has two parts:

**Part 1 (35 pts):** Sea level and salinity in the Chesapeake Bay using a CMEMS ocean reanalysis dataset.

**Part 2 (25 pts):** Country-level environmental and economic indicators.

In [ ]:
# Print your name
print('Sina Shamsian')

---
# Part 1.0 Chesapeake Bay Sea Level and Salinity (35 pts)

**Sea level** is the average height of the ocean surface relative to a reference, not sea level anomaly, thus there are mostly negative values. In a coastal estuary like the Chesapeake Bay, sea level varies on many timescales: from tides over hours, to seasonal cycles driven by thermal expansion and wind patterns over months, to long-term rise driven by ice melt and ocean warming over decades. The Chesapeake region is one of the fastest-rising stretches of coastline in the United States.

**Salinity** measures the dissolved salt concentration in seawater, expressed in practical salinity units (PSU). Open ocean salinity is roughly 35 PSU. In an estuary, salinity drops sharply where freshwater rivers meet the ocean. The Chesapeake Bay receives freshwater from the Susquehanna, Potomac, James, Rappahannock, and York rivers. The salinity gradient from the head of the bay (low salinity) to the mouth (oceanic salinity) is one of the defining features of the system, and changes in river discharge directly modulate this gradient.

This dataset you are provided contains sea surface height and surface salinity, your task is to analyze how they covary.

## Dataset Information

| Field | Details |
|-------|---------|
| Source | CMEMS GLORYS12V1 Global Ocean Physics Reanalysis |
| Variables | `zos` (sea surface height, meters) and `so` (salinity at 0.494 m, PSU) |
| Resolution | 1/12° (~8 km), daily |
| Coverage | 36°N to 40°N, -77.5° to -75°E |
| Time | 1993-01-01 to 2025-12-31 |
| File | `cmems_mod_glo_phy_my_9km_Chesapeake.nc` |


---
Before answering the questions below, load the NetCDF file and inspect the data carefully. A shapefile of major rivers is provided as `chesapeake_major_rivers.shp` (downloaded from Natural Earth). Overlay the major rivers feeding into the Chesapeake Bay onto all your spatial maps.

## Tasks

**Q1: Mean spatial maps with rivers overlaid (3 pts):** Compute the time-averaged **spatial field** for both sea surface height and salinity over the full record. Plot them side by side as subplots. Overlay the major rivers from the provided shapefile and coastlines on each map.

**Q2: Spatial-mean time series with dual axes (3 pts):** Compute the monthly spatial-mean **time series** for both sea surface height and surface salinity. Plot both time series on the same figure using two y-axes (sea level on the left, salinity on the right). Use distinct colors.

**Q3: Linear trend estimation (5 pts):** Decompose the spatial-mean **time series** above into its trend, seasonal and residual components and fit a linear trend to both the sea level and the salinity trend data. Report the sea level trend in mm per year and the salinity trend in PSU per year. Print the slope, intercept, p-value, and R² for each.

**Q4: Spatial Climatology and Anomaly (7 pts):** Compute the **spatial monthly climatology** and the **spatial monthly anomaly** for both sea level and salinity. From these, produce **four final maps (12 subplots for each map)**: (1) the climatology map of sea level, (2) the climatology map of salinity, (3) the anomaly map of sea level, and (4) the anomaly map of salinity.

**Q5: Temporal Anomaly (4 pts):** Compute and plot the **time series anomalies** of the spatial-mean time series using any two baselines of your choice.

**Q6: Correlation and Hovmöller (5 pts):** Please answer either Q6a or Q6b, not both.

***Q6a:*** Compute and plot the spatial correlation map of sea level **trend** and salinity **trend**. *Note:* each gridded time series in the dataset needs to be decomposed and the correlation of the trend estimated.

***Q6b:*** Plot the *monthly* Hovmöller map (months on y axis and longitude on x-axis) of sea level and salinity.

**Q7: Interpretation (8 pts):** In a short paragraph (4–6 sentences), provide a coherent interpretation of your results by synthesizing the trends, anomalies, and decomposition analyses. Reference the figures you created where appropriate. Your response should address the following in a unified discussion:

- How the observed sea level trend in the Chesapeake Bay compares to the global mean rate ((~3.5 mm/yr), and what this suggests about regional influences.
- What the relationship between sea level and salinity reveals about the evolving hydrographic conditions in the bay.
- How river discharge from major tributaries contributes to both the salinity variability and the sea level signal across seasonal and longer timescales.

In [ ]:
### ADD YOUR CODE HERE

## Question 1
import os 
import numpy as np                          
import matplotlib.pyplot as plt            
import xarray as xr
import cartopy.crs as ccrs                              # map projections
import cartopy.feature as cfeature                      # coastlines and land
from cartopy.feature import NaturalEarthFeature
import geopandas as gpd                                 # world boundaries
from shapely.geometry import shape                       # file and folder navigation
data_folder = '/Users/sina/Desktop/ESS_118/datasets/'           # folder where all datasets are stored
os.listdir(data_folder) 

ds = xr.open_dataset(data_folder + 'cmems_mod_glo_phy_my_9km_Chesapeake.nc')
ssh = ds['zos']
sal = ds['so']
mean_ssh = ssh.mean(dim='time',skipna=True)
mean_sal = sal.mean(dim='time',skipna=True)
mean_ssh

# Load rivers shapefile
rivers = gpd.read_file(data_folder + 'chesapeake_major_rivers.shp')

# Create side-by-side maps
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), 
                                subplot_kw={'projection': ccrs.PlateCarree()})

# Plot mean sea surface height
mean_ssh.plot(ax=ax1, cmap='RdBu_r', transform=ccrs.PlateCarree())
ax1.coastlines()
rivers.plot(ax=ax1, color='cyan', linewidth=2, transform=ccrs.PlateCarree())
ax1.set_ylabel('Latitude', fontsize=12)
ax1.set_xlabel('Longitude', fontsize=12)
ax1.set_title('Mean Sea Surface Height (m)')
ax1.gridlines()

# Plot mean salinity
mean_sal.plot(ax=ax2, cmap='viridis', transform=ccrs.PlateCarree())
ax2.coastlines()
rivers.plot(ax=ax2, color='cyan', linewidth=2, transform=ccrs.PlateCarree())
ax2.set_title('Mean Salinity (PSU)')
ax2.gridlines()

plt.tight_layout()
plt.show()





In [ ]:
### ADD YOUR CODE HERE

## Question 2
ssh_mean_time = ssh.mean(dim=['latitude', 'longitude'], skipna=True)   # spatial mean
sal_mean_time = sal.mean(dim=['latitude', 'longitude'], skipna=True)   # spatial mean
# Resample to monthly means
ssh_monthly = ssh_mean_time.resample(time='MS').mean()
sal_monthly = sal_mean_time.resample(time='MS').mean()

# Create figure with dual y-axes
fig, ax1 = plt.subplots(figsize=(14, 5))

# Plot sea surface height on left y-axis
color1 = 'tab:blue'
ax1.set_xlabel('Time')
ax1.set_ylabel('Sea Surface Height (m)', color=color1)
line1 = ax1.plot(ssh_monthly.time, ssh_monthly.values, color=color1, linewidth=1.5, label='SSH')
ax1.tick_params(axis='y', labelcolor=color1)

# Create right y-axis for salinity
ax2 = ax1.twinx()
color2 = 'tab:red'
ax2.set_ylabel('Salinity (PSU)', color=color2)
line2 = ax2.plot(sal_monthly.time, sal_monthly.values, color=color2, linewidth=1.5, label='Salinity')
ax2.tick_params(axis='y', labelcolor=color2)

# Title and legend
fig.suptitle('Monthly Spatial-Mean Time Series: Sea Surface Height & Salinity', fontsize=14, fontweight='bold')
lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper left')

fig.tight_layout()
plt.show()





In [ ]:
### ADD YOUR CODE HERE

## Question 3
from scipy import stats
from statsmodels.tsa.seasonal import seasonal_decompose
import pandas as pd 
# Decompose SSH time series
# Convert xarray to pandas Series for decomposition
ssh_monthly_pd = ssh_monthly.to_pandas()
sal_monthly_pd = sal_monthly.to_pandas()

# Decompose SSH time series
ssh_decomposition = seasonal_decompose(ssh_monthly_pd, model='additive', period=12)
ssh_trend = pd.Series(ssh_decomposition.trend)
ssh_seasonal = pd.Series(ssh_decomposition.seasonal)
ssh_residual = pd.Series(ssh_decomposition.resid)

# Decompose Salinity time series
sal_decomposition = seasonal_decompose(sal_monthly_pd, model='additive', period=12)
sal_trend = pd.Series(sal_decomposition.trend)
sal_seasonal = pd.Series(sal_decomposition.seasonal)
sal_residual = pd.Series(sal_decomposition.resid)


# plot decompositions 
fig, axes = plt.subplots(4,2,figsize=(14,10))

# SSH decomposition
axes[0, 0].plot(ssh_monthly.time.values, ssh_monthly.values, color='blue')
axes[0, 0].set_ylabel('Original SSH (m)')
axes[0, 0].set_title('Sea Surface Height Decomposition')
axes[0, 0].grid()

axes[1, 0].plot(ssh_trend.index, ssh_trend.values, color='darkblue')
axes[1, 0].set_ylabel('Trend (m)')
axes[1, 0].grid()

axes[2, 0].plot(ssh_seasonal.index, ssh_seasonal.values, color='green')
axes[2, 0].set_ylabel('Seasonal (m)')
axes[2, 0].grid()

axes[3, 0].plot(ssh_residual.index, ssh_residual.values, color='red')
axes[3, 0].set_ylabel('Residual (m)')
axes[3, 0].set_xlabel('Time')
axes[3, 0].grid()

# Salinity decomposition
axes[0, 1].plot(sal_monthly.time.values, sal_monthly.values, color='red')
axes[0, 1].set_ylabel('Original Salinity (PSU)')
axes[0, 1].set_title('Salinity Decomposition')
axes[0, 1].grid()

axes[1, 1].plot(sal_trend.index, sal_trend.values, color='darkred')
axes[1, 1].set_ylabel('Trend (PSU)')
axes[1, 1].grid()

axes[2, 1].plot(sal_seasonal.index, sal_seasonal.values, color='orange')
axes[2, 1].set_ylabel('Seasonal (PSU)')
axes[2, 1].grid()

axes[3, 1].plot(sal_residual.index, sal_residual.values, color='purple')
axes[3, 1].set_ylabel('Residual (PSU)')
axes[3, 1].set_xlabel('Time')
axes[3, 1].grid()

plt.tight_layout()
plt.show()


# Remove NaN values for trend fitting
ssh_trend_clean = ssh_trend.dropna()
sal_trend_clean = sal_trend.dropna()

# Convert time to numeric (months since start)
time_numeric_ssh = np.arange(len(ssh_trend_clean))
time_numeric_sal = np.arange(len(sal_trend_clean))

# Fit linear trend to SSH
slope_ssh, intercept_ssh, r_value_ssh, p_value_ssh, std_err_ssh = stats.linregress(time_numeric_ssh, ssh_trend_clean.values)

# Fit linear trend to Salinity
slope_sal, intercept_sal, r_value_sal, p_value_sal, std_err_sal = stats.linregress(time_numeric_sal, sal_trend_clean.values)

# Convert SSH slope from m/month to mm/year
slope_ssh_mm_per_year = slope_ssh * 12 * 1000

# Convert Salinity slope from PSU/month to PSU/year
slope_sal_psu_per_year = slope_sal * 12

# Print results
print("=" * 60)
print("SEA SURFACE HEIGHT (SSH) TREND ANALYSIS")
print("=" * 60)
print(f"Slope: {slope_ssh:.6e} m/month ({slope_ssh_mm_per_year:.4f} mm/year)")
print(f"Intercept: {intercept_ssh:.6f} m")
print(f"P-value: {p_value_ssh:.6e}")
print(f"R²: {r_value_ssh**2:.6f}")
print()

print("=" * 60)
print("SALINITY TREND ANALYSIS")
print("=" * 60)
print(f"Slope: {slope_sal:.6e} PSU/month ({slope_sal_psu_per_year:.6f} PSU/year)")
print(f"Intercept: {intercept_sal:.6f} PSU")
print(f"P-value: {p_value_sal:.6e}")
print(f"R²: {r_value_sal**2:.6f}")
print()

# Plot trend fits
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# SSH trend with fit
ax1.scatter(time_numeric_ssh, ssh_trend_clean.values, alpha=0.6, label='Trend', s=20)
ax1.plot(time_numeric_ssh, intercept_ssh + slope_ssh * time_numeric_ssh, 'r-', linewidth=2, label='Linear Fit')
ax1.set_xlabel('Months since start')
ax1.set_ylabel('SSH (m)')
ax1.set_title(f'SSH Trend: {slope_ssh_mm_per_year:.2f} mm/year (R² = {r_value_ssh**2:.4f})')
ax1.grid()
ax1.legend()

# Salinity trend with fit
ax2.scatter(time_numeric_sal, sal_trend_clean.values, alpha=0.6, label='Trend', color='orange', s=20)
ax2.plot(time_numeric_sal, intercept_sal + slope_sal * time_numeric_sal, 'r-', linewidth=2, label='Linear Fit')
ax2.set_xlabel('Months since start')
ax2.set_ylabel('Salinity (PSU)')
ax2.set_title(f'Salinity Trend: {slope_sal_psu_per_year:.6f} PSU/year (R² = {r_value_sal**2:.4f})')
ax2.grid()
ax2.legend()

plt.tight_layout()
plt.show()




In [ ]:
### ADD YOUR CODE HERE

## Question 4
# Compute monthly climatology (mean for each month across all years)
ssh_climatology = ssh.groupby('time.month').mean(dim='time', skipna=True)
sal_climatology = sal.groupby('time.month').mean(dim='time', skipna=True)

# Compute monthly anomaly (subtract climatology from original data)
ssh_anomaly = ssh.groupby('time.month') - ssh_climatology
sal_anomaly = sal.groupby('time.month') - sal_climatology

# Function to create 12-subplot map figure
def create_monthly_maps(data, title, cmap, units, rivers, is_anomaly=False):
    fig = plt.figure(figsize=(18, 14))
    months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    
    for month in range(1, 13):
        ax = fig.add_subplot(4, 3, month, projection=ccrs.PlateCarree())
        
        # Plot data
        if is_anomaly:
            # For anomaly, select all times with this month using groupby month
            month_data = data.groupby('time.month').mean(dim='time', skipna=True).isel(month=month-1)
        else:
            # For climatology, use isel on month dimension
            month_data = data.isel(month=month-1)
        
        im = month_data.plot(ax=ax, cmap=cmap, transform=ccrs.PlateCarree(), 
                             add_colorbar=False, vmin=data.min().values, vmax=data.max().values)
        
        # Add coastlines and rivers
        ax.coastlines()
        rivers.plot(ax=ax, color='cyan', linewidth=1, transform=ccrs.PlateCarree())
        
        # Formatting
        ax.set_title(f'{months[month-1]}', fontsize=12, fontweight='bold')
        ax.gridlines(draw_labels=False, alpha=0.3)
    
    # Add colorbar
    cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
    cbar = plt.colorbar(im, cax=cbar_ax)
    cbar.set_label(units, fontsize=12)
    
    fig.suptitle(title, fontsize=16, fontweight='bold', y=0.995)
    plt.subplots_adjust(left=0.05, right=0.9, top=0.96, bottom=0.05)
    plt.show()

# Create all four maps
create_monthly_maps(ssh_climatology, 'SSH Climatology (1993-2025)', 'RdBu_r', 'Sea Surface Height (m)', rivers, is_anomaly=False)
create_monthly_maps(sal_climatology, 'Salinity Climatology (1993-2025)', 'viridis', 'Salinity (PSU)', rivers, is_anomaly=False)
create_monthly_maps(ssh_anomaly, 'SSH Monthly Anomaly', 'RdBu_r', 'SSH Anomaly (m)', rivers, is_anomaly=True)
create_monthly_maps(sal_anomaly, 'Salinity Monthly Anomaly', 'RdBu_r', 'Salinity Anomaly (PSU)', rivers, is_anomaly=True)

In [ ]:
### ADD YOUR CODE HERE

## Question 5
# Baseline 1: 1993-2000 (first 7 years)
baseline1_start = '1993-01-01'
baseline1_end = '2000-12-31'

ssh_baseline1 = ssh_monthly.sel(time=slice(baseline1_start,baseline1_end)).mean()
sal_baseline1 = sal_monthly.sel(time=slice(baseline1_start,baseline1_end)).mean()

ssh_anomaly_baseline1 = ssh_monthly - ssh_baseline1
sal_anomaly_baseline1 = sal_monthly - sal_baseline1

# Baseline 2: 2010-2017 (middle period)
baseline2_start = '2010-01-01'
baseline2_end = '2017-12-31'

ssh_baseline2 = ssh_monthly.sel(time=slice(baseline2_start,baseline2_end)).mean()
sal_baseline2 = sal_monthly.sel(time=slice(baseline2_start,baseline2_end)).mean()

ssh_anomaly_baseline2 = ssh_monthly - ssh_baseline2
sal_anomaly_baseline2 = sal_monthly - sal_baseline2

# Create figure with 4 subplots
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Plot 1: SSH anomaly - Baseline 1 (1993-2000)
axes[0, 0].plot(ssh_anomaly_baseline1.time, ssh_anomaly_baseline1.values, color='blue', linewidth=1.5)
axes[0, 0].axhline(y=0, color='k', linestyle='--', alpha=0.3)
axes[0, 0].fill_between(ssh_anomaly_baseline1.time.values, ssh_anomaly_baseline1.values, 0, alpha=0.3, color='blue')
axes[0, 0].set_ylabel('SSH Anomaly (m)', fontsize=11)
axes[0, 0].set_title('SSH Anomaly relative to 1993-2000 baseline', fontsize=12, fontweight='bold')
axes[0, 0].grid(alpha=0.3)

# Plot 2: Salinity anomaly - Baseline 1 (1993-2000)
axes[0, 1].plot(sal_anomaly_baseline1.time, sal_anomaly_baseline1.values, color='red', linewidth=1.5)
axes[0, 1].axhline(y=0, color='k', linestyle='--', alpha=0.3)
axes[0, 1].fill_between(sal_anomaly_baseline1.time.values, sal_anomaly_baseline1.values, 0, alpha=0.3, color='red')
axes[0, 1].set_ylabel('Salinity Anomaly (PSU)', fontsize=11)
axes[0, 1].set_title('Salinity Anomaly relative to 1993-2000 baseline', fontsize=12, fontweight='bold')
axes[0, 1].grid(alpha=0.3)

# Plot 3: SSH anomaly - Baseline 2 (2010-2017)
axes[1, 0].plot(ssh_anomaly_baseline2.time, ssh_anomaly_baseline2.values, color='darkblue', linewidth=1.5)
axes[1, 0].axhline(y=0, color='k', linestyle='--', alpha=0.3)
axes[1, 0].fill_between(ssh_anomaly_baseline2.time.values, ssh_anomaly_baseline2.values, 0, alpha=0.3, color='darkblue')
axes[1, 0].set_ylabel('SSH Anomaly (m)', fontsize=11)
axes[1, 0].set_xlabel('Time', fontsize=11)
axes[1, 0].set_title('SSH Anomaly relative to 2010-2017 baseline', fontsize=12, fontweight='bold')
axes[1, 0].grid(alpha=0.3)

# Plot 4: Salinity anomaly - Baseline 2 (2010-2017)
axes[1, 1].plot(sal_anomaly_baseline2.time, sal_anomaly_baseline2.values, color='darkred', linewidth=1.5)
axes[1, 1].axhline(y=0, color='k', linestyle='--', alpha=0.3)
axes[1, 1].fill_between(sal_anomaly_baseline2.time.values, sal_anomaly_baseline2.values, 0, alpha=0.3, color='darkred')
axes[1, 1].set_ylabel('Salinity Anomaly (PSU)', fontsize=11)
axes[1, 1].set_xlabel('Time', fontsize=11)
axes[1, 1].set_title('Salinity Anomaly relative to 2010-2017 baseline', fontsize=12, fontweight='bold')
axes[1, 1].grid(alpha=0.3)

fig.suptitle('Temporal Anomalies: Two Baseline Comparison', fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()



In [ ]:

### ADD YOUR CODE HERE
from scipy.stats import pearsonr
# Question 6
lats = ssh['latitude'].values 
lons = ssh['longitude'].values
n_time = len(ssh['time'])

print(f'Grid: {len(lats)} latitudes x {len(lons)} longitudes = {len(lats)*len(lons)} pixels')

ssh_trend_map = np.full((len(lats), len(lons),n_time), np.nan)
sal_trend_map = np.full((len(lats),len(lons), n_time), np.nan)

for i, lat in enumerate(lats):
    for j, lon in enumerate(lons):
        
        # SSH
        ssh_series = ssh.sel(latitude=lat, longitude=lon).values
        if not np.all(np.isnan(ssh_series)):
            ssh_pd = pd.Series(ssh_series,
                              index=pd.date_range(start='1993-01-01', periods=n_time, freq='D'))
            try:
                decomp = seasonal_decompose(ssh_pd.dropna(), model='additive', period=365)
                trend = decomp.trend.values
                ssh_trend_map[i, j, :len(trend)] = trend
            except:
                pass
        
        # Salinity
        sal_series = sal.sel(latitude=lat, longitude=lon).values
        if not np.all(np.isnan(sal_series)):
            sal_pd = pd.Series(sal_series,
                              index=pd.date_range(start='1993-01-01', periods=n_time, freq='D'))
            try:
                decomp = seasonal_decompose(sal_pd.dropna(), model='additive', period=365)
                trend = decomp.trend.values
                sal_trend_map[i, j, :len(trend)] = trend
            except:
                pass
    
    if (i + 1) % 2 == 0:
        print(f"  Processed latitude {i + 1}/{len(lats)}")

print("Decomposition complete!\n")   

# Pixel-wise correlation on trend components 
r_map = np.full((len(lats), len(lons)), np.nan)
p_map = np.full((len(lats), len(lons)), np.nan)

for i , lat in enumerate(lats):
    for j, lon in enumerate(lons):
        ssh_trend = ssh_trend_map[i, j, :]
        sal_trend = sal_trend_map[i, j, :]
        valid = ~np.isnan(ssh_trend) & ~np.isnan(sal_trend)
        if valid.sum() > 50:
            r, p = pearsonr(ssh_trend[valid], sal_trend[valid])
            r_map[i, j] = r
            p_map[i, j] = p

print("Correlation computation complete!\n")

# Wrap as data arrays
r_da = xr.DataArray(r_map,
                    coords={'latitude': lats, 'longitude': lons},
                    dims=['latitude', 'longitude'])

p_da = xr.DataArray(p_map,
                    coords={'latitude': lats, 'longitude': lons},
                    dims=['latitude', 'longitude'])

r_sig = r_da.where(p_da < 0.05)  # significant pixels only

# Plot with coordinates of non-significant pixels for dot overlay
nonsig_mask = (p_map >= 0.05) & ~np.isnan(p_map)
lon_grid, lat_grid = np.meshgrid(lons, lats)
nonsig_lons = lon_grid[nonsig_mask]
nonsig_lats = lat_grid[nonsig_mask]

fig, axes = plt.subplots(1, 2, figsize=(16, 6), subplot_kw={'projection': ccrs.PlateCarree()})

# Left panel -- full correlation map
r_da.plot(ax=axes[0], x='longitude', y='latitude',
          cmap='RdBu_r', vmin=-1, vmax=1, transform=ccrs.PlateCarree(),
          cbar_kwargs={'label': 'Pearson r'})
axes[0].coastlines()
rivers.plot(ax=axes[0], color='cyan', linewidth=1, transform=ccrs.PlateCarree(), zorder=2)
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')
axes[0].set_title('SSH-Salinity Trend Correlation (All pixels)')



# Right panel -- significant in color, non-significant as tiny black dots
r_sig.plot(ax=axes[1], x='longitude', y='latitude',
           cmap='RdBu_r', vmin=-1, vmax=1,transform=ccrs.PlateCarree(),
           cbar_kwargs={'label': 'Pearson r'})
axes[1].scatter(nonsig_lons, nonsig_lats,
                color='black', s=1.5, alpha=0.5, zorder=3)
axes[1].coastlines()
rivers.plot(ax=axes[1], color='cyan', linewidth=1, transform=ccrs.PlateCarree(), zorder=2)
axes[1].set_title('Significant (p < 0.05) vs. non-significant (dots)')
axes[1].set_xlabel('Longitude')
axes[1].set_ylabel('Latitude')



plt.suptitle('SSH-Salinity Trend Spatial Correlation (1993-2025)')
plt.tight_layout()
plt.show()

# Print statistics
valid_r = r_map[~np.isnan(r_map)]
sig_count = np.sum(p_map < 0.05)
pos_corr = np.sum((r_map > 0) & (p_map < 0.05))
neg_corr = np.sum((r_map < 0) & (p_map < 0.05))

print("=" * 80)
print("SPATIAL CORRELATION STATISTICS: SSH TREND vs SALINITY TREND")
print("=" * 80)
print(f"Total pixels with valid correlations: {len(valid_r)}")
print(f"Statistically significant (p < 0.05): {sig_count}")
print(f"  Positive correlations (SSH↑ & Sal↑): {pos_corr}")
print(f"  Negative correlations (SSH↑ & Sal↓): {neg_corr}")
print(f"\nCorrelation statistics (all valid pixels):")
print(f"  Mean r:   {np.nanmean(valid_r):>8.4f}")
print(f"  Std r:    {np.nanstd(valid_r):>8.4f}")
print(f"  Min r:    {np.nanmin(valid_r):>8.4f}")
print(f"  Max r:    {np.nanmax(valid_r):>8.4f}")
print("=" * 80)


In [ ]:
### ADD YOUR CODE HERE

## Question 7
#- How the observed sea level trend in the Chesapeake Bay compares to the global mean rate ((~3.5 mm/yr), and what this suggests about regional influences.
# What the relationship between sea level and salinity reveals about the evolving hydrographic conditions in the bay.
# How river discharge from major tributaries contributes to both the salinity variability and the sea level signal across seasonal and longer timescales.
""" The observed sea level trend in the Chesapeake bay, seems to be at around 2.8 mm/yr which is slightly lower than the global mean rate of ~3.5 mm/yr. This suggests that while the Chesapeake Bay is experiencing sea level rise, it may be influenced by regional factors such as land subsidence, sediment compaction, and local oceanographic conditions that could be moderating the rate of sea level rise compared to the global average.
The relationship between sea level and salinity becomes more apparent when I plotted the spatial mean of sea surface height and salinity and found that in areas where there was low salinity there also happened to be high sea surface height. It also seems like the the higher the SSH is the lower the salinity is. This could be be due to the fact that 
when there is a high sea surface height, it could be indicative of increased freshwater input from rivers which would lower the salinity. Conversely, lower sea surface height could indicate less freshwater input and therefore higher salinity. I think that overtime as climate change and sea level rise worsens the increase in sea surface height in the chesapeake bay will lead to more frequent and severe saltwater intrusion events which will have significant impacts on the bay's ecosystem and the communities that rely on it. The increased freshwater input from rivers during high sea surface height events can also lead to more pronounced stratification in the water column, which can affect oxygen levels and nutrient distribution, further impacting marine life in the bay.
"""


---
# Part 2.0 National Environmental and Social Indicators (25 pts)

In this part, you will analyze country-level dataset of seven environmental and economic indicators. Each row is a country and each column is a different variable. Below are some information about the dataset

## Dataset Information

| Field | Details |
|-------|---------|
| Source | World Bank Open Data (averaged over the 21st Century) |
| Rows | ~214 countries |
| File | `country_indicators_final.csv` |

| Variable | Description |
|----------|-------------|
| `co2_per_capita` | CO₂ emissions per capita (metric tons) |
| `renewable_pct` | Renewable energy as a percentage of total energy use |
| `forest_pct` | Forest area as a percentage of total land area |
| `pop_density` | Population density (people per km²) |
| `gdp_per_capita` | GDP per capita (current USD) |
| `electricity_access_pct` | Percentage of population with electricity access |
| `temp_c` | Annual mean temperature (°C) |

The dataset also includes `country`, `continent`, `region`, and `iso3`  columns for labeling.

---

Before answering the questions below, load the CSV file and inspect the data carefully. Please ensure you address any missing values across the seven numeric indicators before proceeding.

## Tasks

**Q8: Correlation matrix (3 pts):** Compute and visualize the correlation matrix among the seven numeric indicators as a heatmap with annotated values. Identify and print the two strongest positive correlations and the two strongest negative correlations.

**Q9: PCA, scree plot, and Biplot (7 pts):** Run PCA on all seven indicators. Produce a scree plot showing the explained variance for all components. State how many components you would retain and justify your choice [*Note:* Retain more than is necessary to explain 80% of the data variance]. Make a biplot of **PC2 versus PC1**. Overlay the variable loading vectors as arrows starting at the origin and labeled with the variable names. **[Extra points - 5 points: Color each country by its continent]** 

**Q10: Explore other variability in the PC Space (7 pts):** Make a 2x2 subplot of any other components of your choice [4 plots in total].

**Q11: PCA interpretation (8 pts):** In a short paragraph (5–7 sentences), provide a coherent interpretation of the PCA results by synthesizing information from the scree plot, loadings, and biplots. Your response should address the following in a unified discussion:

- What dominant patterns or gradients are captured in the dataset, and which variables primarily drive these patterns?
- How do countries distribute in the reduced PC space, and what does this reveal about differences in environmental and socioeconomic conditions?
- What broad interpretation can you make about how development, environment, and climate variables relate to one another across countries?

In [ ]:
### ADD YOUR CODE HERE

import seaborn as sns


## Question 8
df = pd.read_csv(data_folder + 'country_indicators_final.csv')
df.head()

# select only the seven numeric indicators 
indicators = ['co2_per_capita', 'renewable_pct', 'forest_pct', 'pop_density', 
              'gdp_per_capita', 'electricity_access_pct', 'temp_c']
df_indicators = df[indicators].copy()

# Handle missing values - drop rows with any NaN 
df_indicators = df_indicators.dropna()

print(f"Data shape after removing NaNs: {df_indicators.shape}")
print(f"Number of countries with complete data: {len(df_indicators)}\n")

corr_matrix = df_indicators.corr()

fig, ax = plt.subplots(figsize=(10,8))
sns.heatmap(corr_matrix, annot=True,fmt='.2f', cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={'label': 'Correlation'}, ax=ax,
            vmin=-1, vmax=1)
ax.set_title('Correlation Matrix: Country Environmental & Economic Indicators', 
             fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

# Find strongest correlations (excluding diagonal)
# Create a copy and set diagonal to NaN to exclude self-correlations
corr_no_diag = corr_matrix.copy()
np.fill_diagonal(corr_no_diag.values.copy(), np.nan)

# Flatten correlation matrix to find extremes
corr_flat = corr_no_diag.values.flatten()
corr_pairs = []

# Get all unique pairs with their correlations
for i in range(len(indicators)):
    for j in range(i+1, len(indicators)):
        corr_pairs.append({
            'var1': indicators[i],
            'var2': indicators[j],
            'correlation': corr_matrix.iloc[i, j]
        })

# Sort by absolute correlation
corr_pairs_sorted = sorted(corr_pairs, key=lambda x: abs(x['correlation']), reverse=True)

# Print results
print("=" * 80)
print("TWO STRONGEST POSITIVE CORRELATIONS")
print("=" * 80)
pos_corr = [c for c in corr_pairs_sorted if c['correlation'] > 0]
for i, c in enumerate(pos_corr[:2], 1):
    print(f"{i}. {c['var1']} ↔ {c['var2']}: r = {c['correlation']:>7.4f}")
print()

print("=" * 80)
print("TWO STRONGEST NEGATIVE CORRELATIONS")
print("=" * 80)
neg_corr = [c for c in corr_pairs_sorted if c['correlation'] < 0]
for i, c in enumerate(neg_corr[:2], 1):
    print(f"{i}. {c['var1']} ↔ {c['var2']}: r = {c['correlation']:>7.4f}")
print()

print("=" * 80)
print("ALL CORRELATIONS (sorted by strength)")
print("=" * 80)
for i, c in enumerate(corr_pairs_sorted[:10], 1):
    direction = "+" if c['correlation'] > 0 else "−"
    print(f"{i:2d}. {c['var1']:25s} ↔ {c['var2']:25s}: r = {direction}{abs(c['correlation']):7.4f}")

In [ ]:
### ADD YOUR CODE HERE

## Question 9
from sklearn.decomposition import PCA
import matplotlib.patches as mpatches
# Manually standardize the data 
df_mean = df_indicators.mean()
df_std = df_indicators.std()
df_standardized = (df_indicators - df_mean) / df_std

pca = PCA()
pca_scores = pca.fit_transform(df_standardized)

# Get explained variance 
exp_var = pca.explained_variance_ratio_
exp_var_cum = np.cumsum(exp_var)

# Find number of components needed for 80% 
n_components_80 = np.searchsorted(exp_var_cum, 0.80) + 1
print(f"Components needed for 80% variance: {n_components_80}")
print(f"Cumulative variance at 80%: {exp_var_cum[n_components_80-1]:.4f}\n")

# ===== SCREE PLOT =====
n_show = 7  # Show all 7 components
modes = np.arange(1, n_show + 1)

fig, ax1 = plt.subplots(figsize=(10, 6))
ax2 = ax1.twinx()

# Individual variance (bars)
ax1.bar(modes, exp_var[:n_show] * 100, color='steelblue', alpha=0.7, label='Individual')

# Cumulative variance (line))
ax2.plot(modes, exp_var_cum[:n_show] * 100, color = 'tomato', linewidth=2.5,marker='o', markersize=8, label='Cumulative')

# Reference line at 80%
ax2.axhline(80, color='green', linestyle='--', linewidth=2, alpha=0.7, label='80% threshold')

ax1.set_xlabel('Principal Component', fontsize=12)
ax1.set_ylabel('Explained Variance (%)', color='steelblue', fontsize=12)
ax2.set_ylabel('Cumulative Explained Variance (%)', color='tomato', fontsize=12)
ax1.set_title('PCA Scree Plot: Country Indicators', fontsize=14, fontweight='bold')
ax1.set_xticks(modes)
ax1.tick_params(axis='y', labelcolor='steelblue')
ax2.tick_params(axis='y', labelcolor='tomato')
ax1.grid(alpha=0.3, axis='y')

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=11, loc='center left')

plt.tight_layout()
plt.show()

# Print variance table
print("=" * 70)
print("VARIANCE EXPLAINED BY EACH COMPONENT")
print("=" * 70)
print(f"{'PC':<4} {'Individual (%)':<20} {'Cumulative (%)':<20}")
print("-" * 70)
for i in range(n_show):
    print(f"PC{i+1:<2} {exp_var[i]*100:>17.2f}  {exp_var_cum[i]*100:>17.2f}")
print("=" * 70)
print(f"\n✓ RECOMMENDATION: Retain 4 components")
print(f"  - Explains {exp_var_cum[3]*100:.2f}% of variance (exceeds 80% threshold)")
print(f"  - Captures dominant environmental & socioeconomic gradients")
print(f"  - Balances variance explained vs. interpretability\n")

# --- BIPLOT: PC1 vs PC2 -----
# get loadings 
loadings = pca.components_[:2].T * np.sqrt(pca.explained_variance_[:2])

# Color map for continents 
df_plot = df.loc[df_indicators.index].copy()
continents = df_plot['continent'].unique()
continent_colors = {cont: color for cont, color in 
                   zip(continents, plt.cm.tab10(np.linspace(0, 1, len(continents))))}
colors = [continent_colors[cont] for cont in df_plot['continent']]

# Create biplot
fig, ax = plt.subplots(figsize=(14, 10))

# Plot country points (colored by continent)
scatter = ax.scatter(pca_scores[:, 0], pca_scores[:, 1], c=colors, alpha=0.6, s=80,
                    edgecolors='black', linewidth=0.5)

# Annotate each country 
country_names = df_plot['country'].values
for idx, country in enumerate(country_names):
    ax.annotate(country, 
                xy=(pca_scores[idx, 0], pca_scores[idx, 1]),
                fontsize=7,
                alpha=0.75,
                ha='center',
                va='center',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.5, edgecolor='none'))


# Plot loading vectors as arrows
arrow_scale = 3.5
for i, indicator in enumerate(indicators):
    ax.arrow(0, 0, loadings[i, 0] * arrow_scale, loadings[i, 1] * arrow_scale,
            head_width=0.15, head_length=0.15, fc='red', ec='darkred', linewidth=2, alpha=0.8)
    
    # Label arrows
    ax.text(loadings[i, 0] * arrow_scale * 1.15, loadings[i, 1] * arrow_scale * 1.15,
           indicator, fontsize=11, fontweight='bold', ha='center', va='center',
           bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7))

# Axes formatting
ax.axhline(y=0, color='k', linestyle='--', linewidth=0.8, alpha=0.3)
ax.axvline(x=0, color='k', linestyle='--', linewidth=0.8, alpha=0.3)
ax.set_xlabel(f'PC1 ({exp_var[0]*100:.1f}% variance)', fontsize=13, fontweight='bold')
ax.set_ylabel(f'PC2 ({exp_var[1]*100:.1f}% variance)', fontsize=13, fontweight='bold')
ax.set_title('PCA Biplot: PC2 vs PC1 (Country Indicators)', fontsize=14, fontweight='bold')
ax.grid(alpha=0.3)

# Legend for continents
legend_patches = [mpatches.Patch(color=color, label=cont, alpha=0.8) 
                 for cont, color in continent_colors.items()]
ax.legend(handles=legend_patches, loc='upper right', fontsize=11, title='Continent', 
         title_fontsize=12, framealpha=0.95)

plt.tight_layout()
plt.show()

# Print loading information
print("=" * 80)
print("LOADING MATRIX (PC1 and PC2)")
print("=" * 80)
print(f"{'Indicator':<25} {'PC1':<15} {'PC2':<15}")
print("-" * 80)
for i, ind in enumerate(indicators):
    print(f"{ind:<25} {pca.components_[0, i]:>14.4f} {pca.components_[1, i]:>14.4f}")
print("=" * 80)



In [ ]:
### ADD YOUR CODE HERE

## Question 10

### ADD YOUR CODE HERE
# I'll show: PC1 vs PC3, PC1 vs PC4, PC2 vs PC3, PC3 vs PC4

fig, axes = plt.subplots(2, 2, figsize=(16, 14))

df_plot = df.loc[df_indicators.index].copy()
continents = df_plot['continent'].unique()
continent_colors = {cont: color for cont, color in 
                   zip(continents, plt.cm.tab10(np.linspace(0, 1, len(continents))))}
colors = [continent_colors[cont] for cont in df_plot['continent']]

# ===== PLOT 1: PC1 vs PC3 =====
ax = axes[0, 0]
ax.scatter(pca_scores[:, 0], pca_scores[:, 2], c=colors, alpha=0.6, s=80,
          edgecolors='black', linewidth=0.5)

# Add loading vectors for PC1 and PC3
loadings_13 = pca.components_[[0, 2]].T * np.sqrt(pca.explained_variance_[[0, 2]])
arrow_scale = 3.5
for i, indicator in enumerate(indicators):
    ax.arrow(0, 0, loadings_13[i, 0] * arrow_scale, loadings_13[i, 1] * arrow_scale,
            head_width=0.12, head_length=0.12, fc='red', ec='darkred', linewidth=1.5, alpha=0.7)
    ax.text(loadings_13[i, 0] * arrow_scale * 1.15, loadings_13[i, 1] * arrow_scale * 1.15,
           indicator, fontsize=9, fontweight='bold', ha='center', va='center',
           bbox=dict(boxstyle='round,pad=0.2', facecolor='yellow', alpha=0.6))

ax.axhline(y=0, color='k', linestyle='--', linewidth=0.8, alpha=0.3)
ax.axvline(x=0, color='k', linestyle='--', linewidth=0.8, alpha=0.3)
ax.set_xlabel(f'PC1 ({exp_var[0]*100:.1f}% variance)', fontsize=11, fontweight='bold')
ax.set_ylabel(f'PC3 ({exp_var[2]*100:.1f}% variance)', fontsize=11, fontweight='bold')
ax.set_title('PC1 vs PC3', fontsize=12, fontweight='bold')
ax.grid(alpha=0.3)

# ===== PLOT 2: PC1 vs PC4 =====
ax = axes[0, 1]
ax.scatter(pca_scores[:, 0], pca_scores[:, 3], c=colors, alpha=0.6, s=80,
          edgecolors='black', linewidth=0.5)

# Add loading vectors for PC1 and PC4
loadings_14 = pca.components_[[0, 3]].T * np.sqrt(pca.explained_variance_[[0, 3]])
for i, indicator in enumerate(indicators):
    ax.arrow(0, 0, loadings_14[i, 0] * arrow_scale, loadings_14[i, 1] * arrow_scale,
            head_width=0.12, head_length=0.12, fc='red', ec='darkred', linewidth=1.5, alpha=0.7)
    ax.text(loadings_14[i, 0] * arrow_scale * 1.15, loadings_14[i, 1] * arrow_scale * 1.15,
           indicator, fontsize=9, fontweight='bold', ha='center', va='center',
           bbox=dict(boxstyle='round,pad=0.2', facecolor='yellow', alpha=0.6))

ax.axhline(y=0, color='k', linestyle='--', linewidth=0.8, alpha=0.3)
ax.axvline(x=0, color='k', linestyle='--', linewidth=0.8, alpha=0.3)
ax.set_xlabel(f'PC1 ({exp_var[0]*100:.1f}% variance)', fontsize=11, fontweight='bold')
ax.set_ylabel(f'PC4 ({exp_var[3]*100:.1f}% variance)', fontsize=11, fontweight='bold')
ax.set_title('PC1 vs PC4', fontsize=12, fontweight='bold')
ax.grid(alpha=0.3)

# ===== PLOT 3: PC2 vs PC3 =====
ax = axes[1, 0]
ax.scatter(pca_scores[:, 1], pca_scores[:, 2], c=colors, alpha=0.6, s=80,
          edgecolors='black', linewidth=0.5)

# Add loading vectors for PC2 and PC3
loadings_23 = pca.components_[[1, 2]].T * np.sqrt(pca.explained_variance_[[1, 2]])
for i, indicator in enumerate(indicators):
    ax.arrow(0, 0, loadings_23[i, 0] * arrow_scale, loadings_23[i, 1] * arrow_scale,
            head_width=0.12, head_length=0.12, fc='red', ec='darkred', linewidth=1.5, alpha=0.7)
    ax.text(loadings_23[i, 0] * arrow_scale * 1.15, loadings_23[i, 1] * arrow_scale * 1.15,
           indicator, fontsize=9, fontweight='bold', ha='center', va='center',
           bbox=dict(boxstyle='round,pad=0.2', facecolor='yellow', alpha=0.6))

ax.axhline(y=0, color='k', linestyle='--', linewidth=0.8, alpha=0.3)
ax.axvline(x=0, color='k', linestyle='--', linewidth=0.8, alpha=0.3)
ax.set_xlabel(f'PC2 ({exp_var[1]*100:.1f}% variance)', fontsize=11, fontweight='bold')
ax.set_ylabel(f'PC3 ({exp_var[2]*100:.1f}% variance)', fontsize=11, fontweight='bold')
ax.set_title('PC2 vs PC3', fontsize=12, fontweight='bold')
ax.grid(alpha=0.3)

# ===== PLOT 4: PC3 vs PC4 =====
ax = axes[1, 1]
ax.scatter(pca_scores[:, 2], pca_scores[:, 3], c=colors, alpha=0.6, s=80,
          edgecolors='black', linewidth=0.5)

# Add loading vectors for PC3 and PC4
loadings_34 = pca.components_[[2, 3]].T * np.sqrt(pca.explained_variance_[[2, 3]])
for i, indicator in enumerate(indicators):
    ax.arrow(0, 0, loadings_34[i, 0] * arrow_scale, loadings_34[i, 1] * arrow_scale,
            head_width=0.12, head_length=0.12, fc='red', ec='darkred', linewidth=1.5, alpha=0.7)
    ax.text(loadings_34[i, 0] * arrow_scale * 1.15, loadings_34[i, 1] * arrow_scale * 1.15,
           indicator, fontsize=9, fontweight='bold', ha='center', va='center',
           bbox=dict(boxstyle='round,pad=0.2', facecolor='yellow', alpha=0.6))

ax.axhline(y=0, color='k', linestyle='--', linewidth=0.8, alpha=0.3)
ax.axvline(x=0, color='k', linestyle='--', linewidth=0.8, alpha=0.3)
ax.set_xlabel(f'PC3 ({exp_var[2]*100:.1f}% variance)', fontsize=11, fontweight='bold')
ax.set_ylabel(f'PC4 ({exp_var[3]*100:.1f}% variance)', fontsize=11, fontweight='bold')
ax.set_title('PC3 vs PC4', fontsize=12, fontweight='bold')
ax.grid(alpha=0.3)

# Add continent legend to bottom right
legend_patches = [mpatches.Patch(color=color, label=cont, alpha=0.8) 
                 for cont, color in continent_colors.items()]
fig.legend(handles=legend_patches, loc='lower center', ncol=5, fontsize=11, 
          title='Continent', title_fontsize=12, framealpha=0.95, bbox_to_anchor=(0.5, -0.02))

fig.suptitle('Exploring PC Space: 2x2 Subplot of PC Combinations', fontsize=15, fontweight='bold', y=0.995)
plt.tight_layout()
plt.subplots_adjust(bottom=0.08)
plt.show()

# Print component variance summary
print("=" * 70)
print("PC VARIANCE SUMMARY FOR Q10")
print("=" * 70)
print(f"PC1: {exp_var[0]*100:>6.2f}% | PC2: {exp_var[1]*100:>6.2f}% | PC3: {exp_var[2]*100:>6.2f}% | PC4: {exp_var[3]*100:>6.2f}%")
print(f"\nCumulative variance explained by first 4 PCs: {exp_var_cum[3]*100:.2f}%")
print("=" * 70)

In [ ]:
### ADD YOUR CODE HERE

## Question 11
# What dominant patterns or gradients are captured in the dataset, and which variables primarily drive these patterns?
#How do countries distribute in the reduced PC space, and what does this reveal about differences in environmental and socioeconomic conditions?
# What broad interpretation can you make about how development, environment, and climate variables relate to one another across countries?
"""it seems like the pattern between gpd per capita and electricity access per capita had the strongest positive correlation. This makes sense becasue countries with higher GDP per capita often have more resources to invest in infrastructure, including electricity access. The second strongest positive correlation was between co2 per capita and gdp per capita, which also makes sense as wealthier countries tend to have higher energy consumption and thus higher CO2 emissions per capita. 
The strongest negative correlation was between renewable energy percentage and electricity access per capita, which indicates that countries that rely more on renewable energy sources tend to have lower overall electricity usage.The second strongest negative correlation was beteween gpd per capita and temperature which suggests that wealthier countries may have more resources to mitigate the impacts of rising temperatures, or that they are located in regions less affected by temperature increases.
It seems like countries in the PC space are distributed in a way that reflects their development status and environmental conditions. For example, countries with high GDP per capita and electricity access tend to cluster together, while those with higher reliance on renewable energy may form a different cluster. This distribution reveals that there are distinct groupings of countries based on their socioeconomic and environmental profiles, which can be influenced by factors such as geographic location, natural resources, and policy choices.
Overall, across countries it seems like the development, environment, and climate variables are interrelated in comple ways that reflect the trade-offs and synergies between economic growth, energy use, and environmental sustainability. Wealthier countries may have more resources to invest in clean energy and climate adaptation, but they also tend to have higher emissions. Conversely, some countries with lower GDP may rely more on renewable energy but face challenges in accessing electricity. Understanding these relationships is crucial for informing global efforts to achieve sustainable development and address climate change."""

**[Extra Credit – 2 pts]**

How would you rate the difficulty of this exam on a scale of 1 (too easy — 
   challenge me more) to 10 (impossible to complete)? What was the most 
   challenging part?

In [ ]:
# I would say this was an 8 in terms of diffculty because making the 12 subplots was hard and doing the PCA biplot was really challenging. I have to say this was probably the most time consuming challenging thing we have done so far 

---
## Submission checklist

- Save and submit this notebook by **Friday 11:59 PM.**
- renamed the file **ESS118_Midterm_LastName_FirstName.ipynb**.
- run all cells in order with no errors
- filled in all written answer cells